# Iteration runs

This notebook automates the testing of different descriptor and selector combinations. 

The same test set and validation set are used for all models. After these are removed from the dataset, the different sampling methods sample the training set from the remaining pool.

It reads from `data/settings/iteration_settings.csv` containing pairs of descriptors and selectors and their kwargs. Results are output to `outputs/results/iteration_results.csv`.

As comparisons, it trains and tests the base model and full high-fidelity model as well.

## 0.1. Imports and load data

In [38]:
import ase.io
import os
from pathlib import Path
import numpy as np
import importlib
import torch
import matplotlib.pyplot as plt
import pandas as pd
import ast
import time
from sklearn.model_selection import train_test_split

from mace import modules
from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.testing import Tester, extract_latent_space
from mace.training import FreezeStrategy, NaiveStrategy, Trainer, initialise_autoencoder

import sampling_methods.descriptors as descriptors
import sampling_methods.selectors as selectors
import utils.training as training

importlib.reload(descriptors)
importlib.reload(selectors)
importlib.reload(training)


<module 'utils.training' from '/home/lim_yt/X-MACE-sampling/utils/training.py'>

In [39]:
MAX_EPOCHS = 10
R_MAX = 5.0
BATCH_SIZE = 16
BASE_LR = 1.0e-3
TRANSFER_LR = 5.0e-4
DEVICE = torch.device("cpu")

# define wrapper classes
trainer = Trainer(
    max_epochs=MAX_EPOCHS, early_stopping=True, patience=15,
    restore_best=True, device=DEVICE, verbose=True,
)
data_builder = AtomDataLoaderBuilder(
    cutoff=R_MAX, energy_key="REF_energy", forces_key="REF_forces"
)
tester = Tester(device=DEVICE)
loss_fn = modules.InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, forces_weight=5.0, dipoles_weight=0.0,
    nacs_weight=0.0, socs_weight=0.0,
).to(DEVICE)


In [40]:
ROOT_PATH = Path.cwd()
DATA_DIR = ROOT_PATH / '../data'
SETTINGS_CSV = DATA_DIR / 'settings' / 'more_descriptor_selector_combinations.csv'
OUTPUT_DIR = ROOT_PATH / '../outputs'
RESULTS_CSV = OUTPUT_DIR / 'results' / 'more_descriptor_selector_combinations_results.csv'

SEED = 42
torch.manual_seed(SEED)

# Load base dataset
BASE_XYZ = DATA_DIR / 'A02_propene_grid_static.xyz'
BASE_N_GEOMETRIES = '500' 
base_atoms_list = ase.io.read(BASE_XYZ, index=f":{BASE_N_GEOMETRIES}")

# Load transfer dataset
TRANSFER_XYZ = DATA_DIR / "casscf_44_propene_full.xyz"
TRANSFER_N_GEOMETRIES = '500'
transfer_atoms_list = ase.io.read(TRANSFER_XYZ, index=f":{TRANSFER_N_GEOMETRIES}")


In [41]:
# Load descriptor and selector pairs from CSV
settings = pd.read_csv(SETTINGS_CSV)


In [42]:
# Prepare to store results
results = []


## 0.2. Split into test, train, valid sets

In [43]:
# get bond lengths and dihedrals
# tested on propene only

desc_matrix = []
bond_lengths = []
dihedrals = []

for atom in base_atoms_list:
    bond_length = descriptors.get_descriptor("bond_lengths",atom)[0]
    dihedral = descriptors.get_descriptor("dihedral",atom)[0]
    
    bond_lengths.append(bond_length)
    dihedrals.append(dihedral)
    desc_matrix.append([bond_length, dihedral])

desc_matrix = np.asarray(desc_matrix)

print("Number of unique bond lengths:", len(np.unique(bond_lengths)))
print("Number of unique dihedral angles:", len(np.unique(dihedrals)))


Number of unique bond lengths: 41
Number of unique dihedral angles: 90


In [44]:
# extract test set
# the same test set is removed from both base and transfer datasets

TEST_SET_FRACTION = 0.1
TEST_SET_SIZE = int(np.floor(int(BASE_N_GEOMETRIES) * TEST_SET_FRACTION))

test_set_idx = selectors.get_selector("uniform_grid", desc_matrix, TEST_SET_SIZE)
base_test_set = [base_atoms_list[i] for i in test_set_idx]
transfer_test_set = [transfer_atoms_list[i] for i in test_set_idx]
print("Test set size:", len(base_test_set))

# remaining geometries are for training and validation
train_valid_set_idx = np.setdiff1d(np.arange(len(transfer_atoms_list)), test_set_idx)
base_train_valid_set = [base_atoms_list[i] for i in train_valid_set_idx]
transfer_train_valid_set = [transfer_atoms_list[i] for i in train_valid_set_idx]


Test set size: 49


In [45]:
# split remaining geometries into train and valid sets
# the same for both base and transfer datasets

SEED = 42 # set as int to get the same split every time
VALID_SET_FRACTION = 0.1 # as a fraction of the total dataset

train_set_idx, valid_set_idx = train_test_split(
    train_valid_set_idx, test_size=VALID_SET_FRACTION/(1-TEST_SET_FRACTION), random_state=SEED, shuffle=True
)

base_train_set = [base_atoms_list[i] for i in train_set_idx]
print("Base train set size:", len(base_train_set))
base_valid_set = [base_atoms_list[i] for i in valid_set_idx]
print("Base valid set size:", len(base_valid_set))

transfer_train_set = [transfer_atoms_list[i] for i in train_set_idx]
print("\nFull high-fidelity train set size:", len(transfer_train_set))
transfer_valid_set = [transfer_atoms_list[i] for i in valid_set_idx]
print("Full high-fidelity valid set size:", len(transfer_valid_set))


Base train set size: 400
Base valid set size: 51

Full high-fidelity train set size: 400
Full high-fidelity valid set size: 51


## 0.3. Train base and full high-fidelity models once

In [46]:
# base model

base_train_loader = data_builder.load(
    base_train_set, batch_size=BATCH_SIZE, shuffle=True
)
base_valid_loader = data_builder.load(
    base_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
base_test_loader = data_builder.load(
    base_test_set, batch_size=BATCH_SIZE, shuffle=False
)
torch.manual_seed(SEED)

base_model = initialise_autoencoder(data_builder.get_metadata(), preset="lightweight")
base_optimizer = torch.optim.Adam(base_model.parameters(), lr=BASE_LR)

base_model, base_history = trainer.train_model(
    base_model, base_train_loader, base_valid_loader, base_optimizer, loss_fn
)

tester.run_test(base_model, base_test_loader)
base_energy_mae = tester.get_energy_mae()
{"best_epoch": base_history["best_epoch"], "test_energy_mae_ev": base_energy_mae}

results.append({'descriptor': 'base', 'selector': 'base', 'best_epoch': base_history["best_epoch"], 'energy_mae': base_energy_mae})


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3

Epoch 001 | train_loss=110.805522 | valid_loss=143.645348
Epoch 002 | train_loss=98.711857 | valid_loss=123.440926
Epoch 003 | train_loss=59.361273 | valid_loss=40.050778
Epoch 004 | train_loss=36.559696 | valid_loss=30.456585
Epoch 005 | train_loss=34.779711 | valid_loss=26.838164
Epoch 006 | train_loss=30.554280 | valid_loss=25.200441
Epoch 007 | train_loss=28.726743 | valid_loss=25.720980
Epoch 008 | train_loss=27.143945 | valid_loss=23.137841
Epoch 009 | train_loss=26.230271 | valid_loss=22.065236
Epoch 010 | train_loss=24.822448 | valid_loss=23.055558


In [47]:
# save model
base_model_filename = "base_model_propene_500_geometries.pt"
base_model_save_path = OUTPUT_DIR / "base_models" / base_model_filename
torch.save(base_model, base_model_save_path)
print(f"Base model saved to {base_model_save_path}")


Base model saved to /home/lim_yt/X-MACE-sampling/notebooks/../outputs/base_models/base_model_propene_500_geometries.pt


In [48]:
# full high-fidelity model

full_train_loader = data_builder.load(
    transfer_train_set, batch_size=BATCH_SIZE, shuffle=True
)
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)
torch.manual_seed(SEED)

full_model = initialise_autoencoder(data_builder.get_metadata(), preset="lightweight")
full_optimizer = torch.optim.Adam(full_model.parameters(), lr=BASE_LR)

full_model, full_history = trainer.train_model(
    full_model, full_train_loader, full_valid_loader, full_optimizer, loss_fn
)

tester.run_test(full_model, full_test_loader)
full_energy_mae = tester.get_energy_mae()
{"best_epoch": full_history["best_epoch"], "test_energy_mae_ev": full_energy_mae}

results.append({'descriptor': 'full', 'selector': 'full', 'best_epoch': full_history["best_epoch"], 'energy_mae': full_energy_mae})


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3

Epoch 001 | train_loss=94.468593 | valid_loss=137.512264
Epoch 002 | train_loss=81.933370 | valid_loss=105.757315
Epoch 003 | train_loss=39.982557 | valid_loss=29.542008
Epoch 004 | train_loss=23.265453 | valid_loss=17.644935
Epoch 005 | train_loss=16.161132 | valid_loss=12.668763
Epoch 006 | train_loss=15.608734 | valid_loss=21.415628
Epoch 007 | train_loss=14.925137 | valid_loss=11.056147
Epoch 008 | train_loss=13.546927 | valid_loss=13.045257
Epoch 009 | train_loss=11.296221 | valid_loss=10.631888
Epoch 010 | train_loss=10.313873 | valid_loss=10.687597


In [49]:
# save model
full_model_filename = "full_model_propene_500_geometries.pt"
full_model_save_path = OUTPUT_DIR / "full_models" / full_model_filename
torch.save(full_model, full_model_save_path)
print(f"Full model saved to {full_model_save_path}")


Full model saved to /home/lim_yt/X-MACE-sampling/notebooks/../outputs/full_models/full_model_propene_500_geometries.pt


In [30]:
# this will overwrite the base model from the previous section
# run this to load the base model from file instead of using previous section

base_model_filename = "base_model_propene_3731_geometries.pt"
base_model_save_path = OUTPUT_DIR / "base_models" / base_model_filename
base_model = torch.load(base_model_save_path, weights_only=False)

base_model.eval()

AutoencoderExcitedMACE(
  (node_embedding): LinearNodeEmbeddingBlock(
    (linear): Linear(2x0e -> 4x0e | 8 weights)
  )
  (radial_embedding): RadialEmbeddingBlock(
    (bessel_fn): BesselBasis(r_max=5.0, num_basis=4, trainable=False)
    (cutoff_fn): PolynomialCutoff(p=3.0, r_max=5.0)
  )
  (perm_encoder): PermutationInvariantEncoder(
    (elementwise_nn): Sequential(
      (0): Linear(in_features=1, out_features=16, bias=True)
      (1): ELU(alpha=1.0)
      (2): Linear(in_features=16, out_features=16, bias=True)
      (3): ELU(alpha=1.0)
      (4): Linear(in_features=16, out_features=16, bias=True)
      (5): ELU(alpha=1.0)
      (6): Linear(in_features=16, out_features=16, bias=True)
      (7): ELU(alpha=1.0)
    )
    (post_aggregation_nn): Sequential(
      (0): Linear(in_features=16, out_features=16, bias=True)
      (1): ELU(alpha=1.0)
      (2): Linear(in_features=16, out_features=16, bias=True)
      (3): ELU(alpha=1.0)
      (4): Linear(in_features=16, out_features=16, bias=

# 1. Iterative testing

In [50]:
# parses descriptor kwargs and selector kwargs in settings file
def parse_kwargs(value):
    if pd.isna(value) or value is None:
        return {}
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return {}
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return value if isinstance(value, dict) else {}

# handles encoded_energies descriptor (needs encoder as a kwarg)
base_encoder = base_model.perm_encoder
def resolve_kwargs(kwargs):
    resolved = {}
    for k, v in kwargs.items():
        if v == "base_encoder":
            resolved[k] = base_encoder
        else:
            resolved[k] = v
    return resolved

def serialize_kwargs(kwargs):
    return {
        k: "base_encoder" if v is base_encoder else v
        for k, v in kwargs.items()
    }

# handles latent_space descriptor
base_train_loader_unshuffle = data_builder.load( # base train loader without shuffle
    base_train_set, batch_size=BATCH_SIZE, shuffle=False
)
latent_space = extract_latent_space(
    base_model, base_train_loader_unshuffle, device=DEVICE
)


In [51]:
# Initialise valid and test loaders
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)


In [52]:
# Iterate over descriptor and selector pairs
for index, row in settings.iterrows():
    start_time = time.time()
    
    descriptor = row['descriptor']
    selector = row['selector']

    descriptor_kwargs = resolve_kwargs(parse_kwargs(row.get('descriptor_kwargs', '')))
    selector_kwargs = resolve_kwargs(parse_kwargs(row.get('selector_kwargs', '')))

    print(f"\n--- Descriptor: {descriptor} | Selector: {selector} ---")
    
    # Run descriptor
    desc_matrix = []
    if descriptor == 'latent_space':
        desc_matrix = latent_space
    else:
        for atom in base_train_set:
            try:
                desc = descriptors.get_descriptor(descriptor, atom, **descriptor_kwargs)
            except (TypeError, ValueError):
                desc = descriptors.get_descriptor(descriptor, atom)
            desc_matrix.append(desc)
        desc_matrix = np.asarray(desc_matrix)

    # Run selector
    try:
        sampled_idx = selectors.get_selector(selector, desc_matrix, 100, **selector_kwargs)
    except (TypeError, ValueError):
        sampled_idx = selectors.get_selector(selector, desc_matrix, 100)

    sampled_atoms = [transfer_train_set[i] for i in sampled_idx]

    # Initialise train loader
    transfer_train_loader = data_builder.load(
        sampled_atoms, batch_size=BATCH_SIZE, shuffle=True
    )

    # Train the model
    transfer_model = NaiveStrategy().apply(base_model)
    transfer_optimizer = torch.optim.Adam(transfer_model.parameters(), lr=TRANSFER_LR)

    torch.manual_seed(SEED)

    transfer_model, transfer_history = trainer.train_model(
        transfer_model, transfer_train_loader, full_valid_loader, transfer_optimizer, loss_fn
    )
    
    # Test the model
    tester = Tester(device=torch.device('cpu'))
    tester.run_test(transfer_model, full_test_loader)
    best_epoch = transfer_history["best_epoch"]
    energy_mae = tester.get_energy_mae()

    duration = round(time.time() - start_time, 2)
    
    print(f"--- Best epoch: {best_epoch} | Energy MAE: {energy_mae} | Duration: {duration}---")
    
    results.append({
        'descriptor': descriptor,
        'selector': selector,
        'descriptor_kwargs': serialize_kwargs(descriptor_kwargs),
        'selector_kwargs': serialize_kwargs(selector_kwargs),
        'best_epoch': best_epoch,
        'energy_mae': energy_mae
    })



--- Descriptor: latent_space | Selector: farthest_point_sampling ---
Epoch 001 | train_loss=20.411043 | valid_loss=14.319821
Epoch 002 | train_loss=16.465962 | valid_loss=11.296827
Epoch 003 | train_loss=14.590576 | valid_loss=12.431364
Epoch 004 | train_loss=13.764895 | valid_loss=11.794213
Epoch 005 | train_loss=12.020142 | valid_loss=10.506390
Epoch 006 | train_loss=10.974374 | valid_loss=10.728921
Epoch 007 | train_loss=11.452322 | valid_loss=9.739580
Epoch 008 | train_loss=10.730735 | valid_loss=9.666722
Epoch 009 | train_loss=10.086055 | valid_loss=9.496499
Epoch 010 | train_loss=10.180518 | valid_loss=9.111600
--- Best epoch: 10 | Energy MAE: 0.718447744846344 | Duration: 93.85---

--- Descriptor: latent_space | Selector: dbscan_weighted ---
n clusters: 1
n clusters with samples: 1
labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [54]:
# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Results saved to {RESULTS_CSV}')

Results saved to /home/lim_yt/X-MACE-sampling/notebooks/../outputs/results/more_descriptor_selector_combinations_results.csv


## 2. Code cells for 1 run

In [13]:
DESCRIPTOR_TYPE = ["bond_lengths", "bond_angles", "dihedral", "energies", "encoded_energies", "soap", "acsf", "mbtr"]
DESCRIPTOR = DESCRIPTOR_TYPE[4]  # choose the descriptor type to use

base_encoder = base_model.perm_encoder

# matrix of descriptors for each geometry
# sampling is done on base dataset, out of the training pool only
# ie test and valid sets are already removed
desc_matrix = []
for atom in base_train_set:
    desc_matrix.append(descriptors.get_descriptor(DESCRIPTOR,atom,base_encoder))
desc_matrix = np.asarray(desc_matrix)

# n*m, where n is the number of geometries and m is the dimension of the descriptor
# eg if we use bond_lengths and propene, m=2 because there's 2 CC bonds in propene
print("desc_matrix shape:", desc_matrix.shape)
print("desc_matrix unique shape:", np.unique(desc_matrix, axis=0).shape)
print("desc_matrix:\n", desc_matrix)


desc_matrix shape: (2995, 16)
desc_matrix unique shape: (2954, 16)
desc_matrix:
 [[ 171.55641174  242.11019897  -31.0217762  ... -258.95349121
  -223.0693512   186.06126404]
 [ 171.60189819  242.17453003  -31.03016853 ... -259.02191162
  -223.12860107  186.11030579]
 [ 171.45986938  241.97351074  -31.00399971 ... -258.80810547
  -222.94322205  185.95709229]
 ...
 [ 171.62950134  242.21359253  -31.03524208 ... -259.06344604
  -223.16471863  186.1401062 ]
 [ 171.39927673  241.88774109  -30.99276924 ... -258.71679688
  -222.86421204  185.89167786]
 [ 171.52253723  242.0622406   -31.01553535 ... -258.90246582
  -223.02511597  186.02470398]]


In [15]:
SELECTOR_TYPE = ["random_sampling", "farthest_point_sampling", "k_means_clustering", "birch", "dbscan", "dbscan_weighted"]
SELECTOR = SELECTOR_TYPE[0]  # choose the selector type to use

# number of samples to select from the transfer dataset
N_SAMPLES = 100

# select samples based on the descriptor matrix
sampled_idx = selectors.get_selector(SELECTOR, desc_matrix, N_SAMPLES)

# sample atoms out of transfer dataset
sampled_atoms = [transfer_train_set[i] for i in sampled_idx]

print("sampled indices:\n", sampled_idx)
print("number sampled:", len(sampled_idx))

sampled indices:
 [2854 1439 2319 1980  192 1189 1834   16 2780 1867 1650 2646  398 2734
  641 1223 2861 1317  706 2084 1613 1673 2024 2489 1378 2129  411 2330
 1151 2117 2751  983    8  251 2620 1442 1323  528  307 2771  492 2200
  401 2078 2662  918 1314 2718 1315  519 2012 2124 1247 1371 1143  402
 2702 2445 1670 2624 1794  845  920 2569  705  139  777 1797 2603  860
 2896  693  792 2846 2506 2136 2923   83 1363  294 2909 1771  146  879
 2340 2171 2222 1082 1564  520 2104  976 2845 1863 2543  878 2213 1816
 2633 1437]
number sampled: 100


In [16]:
transfer_train_loader = data_builder.load(
    sampled_atoms, batch_size=BATCH_SIZE, shuffle=True
)
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)

transfer_model = NaiveStrategy().apply(base_model)
transfer_optimizer = torch.optim.Adam(transfer_model.parameters(), lr=TRANSFER_LR)

torch.manual_seed(SEED)

transfer_model, transfer_history = trainer.train_model(
    transfer_model, transfer_train_loader, full_valid_loader, transfer_optimizer, loss_fn
)

tester.run_test(transfer_model, full_test_loader)
transfer_energy_mae = tester.get_energy_mae()
{"best_epoch": transfer_history["best_epoch"], "test_energy_mae_ev": transfer_energy_mae}


Epoch 001 | train_loss=11.273355 | valid_loss=8.548976
Epoch 002 | train_loss=8.622177 | valid_loss=6.661526
Epoch 003 | train_loss=7.038686 | valid_loss=5.763349
Epoch 004 | train_loss=6.265099 | valid_loss=5.317880
Epoch 005 | train_loss=5.879915 | valid_loss=5.289293
Epoch 006 | train_loss=5.676054 | valid_loss=5.185351
Epoch 007 | train_loss=5.708424 | valid_loss=4.937463
Epoch 008 | train_loss=5.567621 | valid_loss=4.840658
Epoch 009 | train_loss=5.363175 | valid_loss=4.902267
Epoch 010 | train_loss=5.414978 | valid_loss=4.836897
Epoch 011 | train_loss=5.274954 | valid_loss=4.929088
Epoch 012 | train_loss=5.501983 | valid_loss=4.785780
Epoch 013 | train_loss=5.652386 | valid_loss=4.759015
Epoch 014 | train_loss=5.367650 | valid_loss=4.921945
Epoch 015 | train_loss=6.162858 | valid_loss=5.040338
Epoch 016 | train_loss=5.511419 | valid_loss=5.172635
Epoch 017 | train_loss=5.440444 | valid_loss=4.859781
Epoch 018 | train_loss=5.367051 | valid_loss=4.856334
Epoch 019 | train_loss=5.31

{'best_epoch': 44, 'test_energy_mae_ev': 0.6823110580444336}